# L18 · 部署初探：让你的服务上线

**学习目标**
- 理解「部署」：从「我电脑能跑」到「全世界能访问」
- 理解容器(Docker)思想与 `requirements.txt` 的作用
- 体验「一键产出可部署工程包」的工业流程

**前置依赖**：L14-L17（API + DB + 鉴权）  
**预计时长**：40 分钟  
**技术栈**：`fastapi`、`uvicorn`（生成 `requirements.txt`、Dockerfile）

---

## 概念讲解：部署 = 把「我的玩具」变成「公共基础设施」

你写的 API 现在只能在你电脑跑。要让别人用，需要「部署」到一台永远开着的服务器。
现代部署靠两样东西：

- **`requirements.txt`**：列出依赖，别人一键 `pip install -r` 装好环境
- **Docker**：把「代码+环境」打包成一个「集装箱」，到哪都能原样跑（一次构建，处处运行）

这一课我们不真买服务器，而是**产出一套可部署的工程文件**，体验完整交付流程。

## 第一步：写出服务主文件 `main.py`（用代码生成，模拟你交付的代码）

In [ ]:
main_py = '''
from fastapi import FastAPI
app = FastAPI(title="Hello Deploy")

@app.get("/")
def root():
    return {"msg": "🚀 我上线了！这是你的第一个可部署 API"}

@app.get("/health")
def health():
    return {"status": "ok"}
'''
with open("main.py", "w") as f:
    f.write(main_py)
print("✅ 已生成 main.py")

## 第二步：生成 `requirements.txt` 与 `Dockerfile`

In [ ]:
with open("requirements.txt", "w") as f:
    f.write("fastapi==0.110.0\nuvicorn==0.29.0\n")

dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY main.py .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
with open("Dockerfile", "w") as f:
    f.write(dockerfile)
print("✅ 已生成 requirements.txt 和 Dockerfile")
print("  —— 别人拿到这两个文件 + main.py，就能 `docker build` 部署到任何云平台")

# 🎯 AHA 顿悟单元格：本地跑起「生产配置」的服务

运行下面代码。会用 `uvicorn` 以**生产配置**（0.0.0.0 可外部访问）启动 `main.py`，
然后自动请求 `/` 和 `/health`，你会看到服务真的「活」在 8000 端口。
这，就是部署到云上之前，在本地验证的最后一步。

> 你刚刚走完了「写代码 → 列依赖 → 打容器 → 启动验证」的全链路。再往前一步，就是真上云。

In [ ]:
# ===== 运行我！本地以生产配置启动并验证 =====
import uvicorn, threading, time, requests, importlib.util

# 动态加载刚生成的 main.py 作为 app
spec = importlib.util.spec_from_file_location("mainmod", "main.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
PORT = 8000
threading.Thread(target=lambda: uvicorn.run(mod.app, host="0.0.0.0", port=PORT, log_level="warning"), daemon=True).start()
time.sleep(2)

r1 = requests.get(f"http://127.0.0.1:{PORT}/")
r2 = requests.get(f"http://127.0.0.1:{PORT}/health")
print("  GET /        →", r1.json())
print("  GET /health  →", r2.json())
print("  🚀 生产配置服务已在 0.0.0.0:8000 运行（外部可访问）")
print("  📦 交付物：main.py + requirements.txt + Dockerfile —— 可直接 docker build 部署！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：部署抽象（环境一致性）；Docker 集装箱比喻。  
**易错点**：0.0.0.0 vs 127.0.0.1（前者外部可访问）；端口 8000 冲突。  
**AHA 机制**：本地以生产配置跑通自生成的服务，完整交付链路可视化，强「我能上线了」成就。  
**衔接**：阶段四（L19+ 大模型）把 AI 能力塞进这个 API；L30 MLOps 深化部署。  
**依赖**：fastapi/uvicorn；Docker 仅生成文件不执行（环境未必装）。  
**备注**：生成本地 `main.py`/`requirements.txt`/`Dockerfile` 是教学产物，应提示学员这是演示，可清理。

# 📚 作业 / 下一步

1. 把 `main.py` 里的 `/` 返回改成你的名字，重新运行 AHA 单元格。
2. （进阶）若有 Docker，尝试 `docker build -t myapi . && docker run -p 8000:8000 myapi`。
3. 进入 **阶段四 · 大模型应用工程**：L19 Transformer 通识 —— 揭开 ChatGPT 背后的那块积木。